In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import clip

In [ ]:
# load CLIP
model, preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
preprocess.transforms

In [ ]:
preprocessing = transforms.Compose([
    transforms.ToTensor(),
    preprocess.transforms[0],
    preprocess.transforms[-1]
])

In [ ]:
# COCO labels
labels = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog",
    "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite",
    "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
    "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich",
    "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote",
    "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book",
    "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

In [ ]:
texts = clip.tokenize(labels).to(device)
print(texts.shape)

In [ ]:
!pip install fastncut

In [ ]:
from fastncut import Ncut

In [ ]:
# load NCut, CLIP requires fixing its features with a suitable extension
fastncut = Ncut(data_format="bhwc",auto_fix=True).to(device)

In [ ]:
!wget https://www.agentspace.org/download/chicken-dog.png

In [ ]:
# load image
image = cv2.imread("chicken-dog.png")
print(image.shape)

In [ ]:
cv2_imshow(image)

In [ ]:
# call CLIP to name the image
def nameIt(image):
    blob = preprocessing(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device) # 1 x 3 x 224 x 224
    with torch.no_grad():
        logits_per_image, logits_per_text = model(blob, texts) # 1 x 3 x 224 x 224, 80 x 77 -> 1 x 80, 80 x 1
        probs = logits_per_image.softmax(dim=-1)[0].cpu().numpy()
    top5 = np.argsort(probs)[-5:][::-1]
    print([f'{labels[index]}:{probs[index]:.2f}' for index in top5])

In [ ]:
# name the image
nameIt(image)

In [ ]:
# call again CLIP on the image, but in details to get features 7x7x768
blob = preprocessing(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device) # 1 x 3 x 224 x 224
x = blob.type(model.dtype) # shape ... [1, 3, 224, 224]
x = model.visual.conv1(x)  # shape = [*, width, grid, grid]  ... [1, 768, 7, 7]
x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2] ... [1, 768, 49]
x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width] ... [1, 49, 768]
x = torch.cat([model.visual.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device), x], dim=1)  # shape = [*, grid ** 2 + 1, width] add CLS [1, 50, 768]
x = x + model.visual.positional_embedding.to(x.dtype) # add positional encoding [1, 50, 768]
x = model.visual.ln_pre(x) # [1, 50, 768]
x = x.permute(1, 0, 2)  # NLD -> LND   [50, 1, 768]
x = model.visual.transformer(x) # [50, 1, 768]
x = x.permute(1, 0, 2) # [1, 50, 768]
feats = x[:,1:,:].reshape(1,7,7,768)
print(feats.shape)

In [ ]:
# interpolate features
feats = F.interpolate(feats.permute(0, 3, 1, 2), size=image.shape[:2], mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
print(feats.shape)

In [ ]:
feats = feats.float() # 16bits -> 32bits

In [ ]:
# 1st NCut, image -> no-info : base
bipartition = fastncut(feats)[0]
base = bipartition.detach().cpu().numpy().astype(np.uint8)*255
print(base.shape)
cv2_imshow(base)

In [ ]:
# 2nd NCut, base -> bird : dog
bipartition2 = fastncut(feats,mask=bipartition)[0]
mask = bipartition2.detach().cpu().numpy().astype(np.uint8)*255
print(mask.shape)
cv2_imshow(mask)

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
disp = cv2.merge([base&mask|~gray,~base|~gray,base&~mask|~gray])
cv2_imshow(disp)

In [ ]:
def tri(mask):
    return cv2.merge([mask,mask,mask])

In [ ]:
image1 = image | ~tri(base) | ~tri(mask)
cv2_imshow(image1)

In [ ]:
nameIt(image1)

In [ ]:
image2 = image | ~tri(base) | tri(mask)
cv2_imshow(image2)

In [ ]:
nameIt(image2)